# AI-Powered Study Assistant

The AI-Powered Study Assistant is an intelligent system that helps students interact with their study materials efficiently. Using Retrieval-Augmented Generation, it allows users to upload documents and ask questions in natural language. The system processes documents by extracting, chunking, and converting text into embeddings stored in a vector database. When a query is asked, it retrieves relevant content and generates accurate, context-based answers using a language model. This reduces manual searching, saves time, and improves learning. The system is scalable, user-friendly, and can be extended with features like summarization, voice interaction, and personalized recommendations.

## 1. Install and Import Dependencies

In [ ]:
%pip install -r ../requirements.txt

In [ ]:
# If needed, install packages in the notebook environment
# !pip install -r ../requirements.txt

import os
import shutil
from pathlib import Path

from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader, TextLoader, Docx2txtLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

In [ ]:
BASE_DIR = Path("..").resolve()
DATA_DIR = BASE_DIR / "data"
STORE_DIR = BASE_DIR / "store"

DATA_DIR.mkdir(exist_ok=True)
STORE_DIR.mkdir(exist_ok=True)

print("Data folder:", DATA_DIR)
print("Store folder:", STORE_DIR)

In [ ]:
def load_documents(folder_path: Path):
    documents = []

    for file_path in folder_path.rglob("*"):
        if file_path.is_file():
            suffix = file_path.suffix.lower()

            try:
                if suffix == ".pdf":
                    loader = PyPDFLoader(str(file_path))
                    documents.extend(loader.load())
                elif suffix == ".txt":
                    loader = TextLoader(str(file_path), encoding="utf-8")
                    documents.extend(loader.load())
                elif suffix in [".docx"]:
                    loader = Docx2txtLoader(str(file_path))
                    documents.extend(loader.load())
            except Exception as e:
                print(f"Skipping {file_path.name}: {e}")

    return documents

docs = load_documents(DATA_DIR)
print(f"Loaded {len(docs)} document chunks from raw files")

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(docs)
print(f"Created {len(chunks)} chunks")

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully")

In [ ]:
if len(chunks) > 0:
    vectorstore = FAISS.from_documents(chunks, embedding_model)
    vectorstore.save_local(str(STORE_DIR))
    print("Vector store created and saved successfully")
else:
    print("No document chunks available. Add files to the data folder first.")

In [ ]:
if (STORE_DIR / "index.faiss").exists():
    vectorstore = FAISS.load_local(
        str(STORE_DIR),
        embedding_model,
        allow_dangerous_deserialization=True
    )
    print("Vector store loaded successfully")
else:
    print("Vector store not found. Run the previous cell first.")

In [ ]:
def answer_question(question, k=4):
    if 'vectorstore' not in globals():
        return "Vector store is not loaded."

    relevant_docs = vectorstore.similarity_search(question, k=k)

    context = "\n\n".join(
        [f"Source {i+1}:\n{doc.page_content}" for i, doc in enumerate(relevant_docs)]
    )

    prompt = f"""
You are a helpful AI study assistant.
Answer the question only using the provided context.
If the answer is not in the context, say you do not know.

Context:
{context}

Question:
{question}

Answer:
"""

    return prompt

sample_question = "What is Retrieval-Augmented Generation?"
print(answer_question(sample_question))

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, Markdown

    question_input = widgets.Text(
        value="What is Retrieval-Augmented Generation?",
        description="Question:",
        layout=widgets.Layout(width="80%")
    )

    output = widgets.Output()

    def on_submit(change):
        with output:
            output.clear_output()
            q = question_input.value
            print("Question:", q)
            print()
            print(answer_question(q))

    question_input.observe(on_submit, names="value")
    display(question_input, output)

except Exception as e:
    print("ipywidgets not available:", e)

In [ ]:
print("AI-Powered Study Assistant notebook completed.")
print("Flow:")
print("1. Load documents")
print("2. Split into chunks")
print("3. Create embeddings")
print("4. Save to FAISS vector store")
print("5. Retrieve relevant context for user questions")